In [ ]:
    ############    #############   Server-Sent Events (SSE) and WebSocket LLM Streaming   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.2 Backend Engineering
 =>  Week:  Week 1
 =>  Track: Mandatory (Addition)

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   SSE vs WebSocket for LLM Token Streaming   #############   ##############   

 =>  SSE (Server-Sent Events): one-directional, server -> client, over plain HTTP. Perfect
       for streaming LLM tokens: the client sends one request, the server streams the
       response back chunk by chunk. Simpler infra (works through most proxies/load
       balancers unmodified), auto-reconnect built into the browser's EventSource API.

 =>  WebSocket: full bidirectional, persistent connection. Needed when the CLIENT also
       needs to send messages mid-stream (e.g. 'stop generating', a multi-turn voice
       interface) -- otherwise it's more infra complexity than an LLM token stream needs.

 =>  Default to SSE for 'stream an LLM response to the UI'; reach for WebSocket only when
       you have a genuine bidirectional need.


In [ ]:
import asyncio
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

app = FastAPI()

async def fake_llm_tokens(prompt: str):
    tokens = ["The", " quick", " brown", " fox"]
    for token in tokens:
        await asyncio.sleep(0.01)
        yield token

@app.get("/stream")
async def stream(prompt: str = "hello"):
    async def event_generator():
        async for token in fake_llm_tokens(prompt):
            yield f"data: {token}\n\n"   # SSE wire format: 'data: <payload>\n\n'
        yield "data: [DONE]\n\n"
    return StreamingResponse(event_generator(), media_type="text/event-stream")

client = TestClient(app)
with client.stream("GET", "/stream", params={"prompt": "hi"}) as response:
    print("status:", response.status_code)
    print("content-type:", response.headers["content-type"])
    for chunk in response.iter_text():
        print(repr(chunk))


In [ ]:
 =>  Each 'data: ...\n\n' block is one SSE event -- the blank line (\n\n) is what tells
       the client 'this event is complete'. A browser's EventSource API parses this format
       natively; a custom frontend can also just read the raw stream.

 =>  media_type='text/event-stream' is what makes this SSE rather than a generic chunked
       response -- it sets the right Content-Type and disables response buffering on most
       proxies.

 =>  Combine this with 'await request.is_disconnected()' (see Phase 0.1's Async Client
       Disconnection notebook) to stop generating -- and stop paying for -- tokens once the
       client has gone away.


In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient

app_ws = FastAPI()

@app_ws.websocket("/ws/chat")
async def chat_socket(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            user_message = await websocket.receive_text()
            for token in ["Echo", ": ", user_message]:
                await websocket.send_text(token)
    except WebSocketDisconnect:
        pass  # client closed the connection

client = TestClient(app_ws)
with client.websocket_connect("/ws/chat") as websocket:
    websocket.send_text("hello there")
    print(websocket.receive_text())
    print(websocket.receive_text())
    print(websocket.receive_text())


In [ ]:
 =>  Unlike the SSE example, the client here can send a NEW message on the same open
       connection at any time -- that bidirectional capability is WebSocket's whole reason
       for existing; if you never need the client to send more than the one initial
       request, SSE is simpler and sufficient.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Wire fake_llm_tokens up to a real streaming LLM API call (OpenAI/Anthropic
           streaming mode) instead of a hardcoded token list.

 =>  [ ] Add 'await request.is_disconnected()' inside event_generator and confirm (via a
           print/log) that generation stops early on a simulated disconnect.

 =>  [ ] Build a minimal HTML page using the browser's native EventSource API against the
           /stream endpoint.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Forgetting the double newline ('\n\n') at the end of each SSE event -- clients will
       hang waiting for the event to 'complete'.

 =>  Using WebSocket for a simple one-shot LLM stream 'just in case' -- it costs you a
       stateful, harder-to-load-balance connection for a capability (bidirectional
       mid-stream messages) you don't actually use.

 =>  Not handling WebSocketDisconnect -- an unhandled disconnect exception can crash the
       handler loop instead of cleanly ending the session.
